# 1. Finding and fetching the data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YOUR-GITHUB-USER/chord-recognition-book/blob/main/notebooks/01_finding_and_fetching_data.ipynb)

> **Paper ↔ code.** Section I-A of the paper names its data: the *RWC Music Database* (Goto et al., 2002/2003) with the *AIST annotations* (Goto, 2006), re-released in 2026 by Balke et al. as an open corpus. This notebook takes you from those citations to files on disk, using `chordlab.data`.

Reading a paper is one thing; getting its data onto your computer is another. This notebook is deliberately slow and explicit about that step, because it is where most students get stuck.

## 1.1 Where do MIR datasets live?

Music datasets are scattered over a few kinds of places:

| Kind of place | Examples | What you usually get |
|---|---|---|
| Research data repositories | [Zenodo](https://zenodo.org) (CERN), figshare | audio/annotation archives with a **DOI**, a **license** and **checksums** |
| Code hosting | GitHub, GitLab | annotations, loaders, small derived files — rarely audio |
| Dataset loaders | [`mirdata`](https://mirdata.readthedocs.io) | uniform Python access, index files with checksums; audio often still needs a separate download |
| Community lists | [ismir.net/resources/datasets](https://ismir.net/resources/datasets/) | a curated table of MIR datasets and what they annotate |

For twenty years RWC was the odd one out: the audio was mailed on physical media for a nominal fee and the annotations lived on static web pages. Since February 2026 the audio sits on Zenodo under **CC BY-NC 4.0** and the annotations are curated on GitHub. That is what makes this book possible.

In [1]:
# --- Setup: works locally (run from anywhere inside the repository) and on Google Colab ---
import os, sys, pathlib, subprocess

REPO = "YOUR-GITHUB-USER/chord-recognition-book"   # <-- change this after forking

if "google.colab" in sys.modules:
    root = pathlib.Path("/content") / REPO.split("/")[1]
    if not root.exists():
        subprocess.run(["git", "clone", "-q", f"https://github.com/{REPO}.git", str(root)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(root)], check=True)
else:
    root = pathlib.Path.cwd().resolve()
    while not (root / "chordlab").exists() and root != root.parent:
        root = root.parent
os.chdir(root)
sys.path.insert(0, str(root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa, librosa.display
import IPython.display as ipd

import chordlab
from chordlab import data, features, templates, hmm, evaluate, midi_chords, synth, plots

plt.rcParams["figure.figsize"] = (11, 3.5)
plt.rcParams["figure.dpi"] = 110
print(f"chordlab {chordlab.__version__} loaded from {root}")
print("data folder:", data.data_root())

chordlab 0.1.0 loaded from C:\Users\uture\Downloads\files\chord-recognition-book\chord-recognition-book
data folder: D:\rwc-data


In [2]:
# Optional (Colab only): keep downloads on Google Drive so they survive the session.
# from google.colab import drive
# drive.mount("/content/drive")
# data.set_data_root("/content/drive/MyDrive/chordlab-data")

## 1.2 Reading a Zenodo record

A Zenodo record is a small contract between the people who published the data and you. Before downloading anything, read three things on the record page:

1. **License** — here *Creative Commons Attribution-NonCommercial 4.0*: you may use and share the data for non-commercial purposes if you credit the authors.
2. **Citation requests** — the record asks you to cite four publications (they are in `references.bib`).
3. **Files and checksums** — each zip has an MD5 hash; verifying it after the download tells you the file is complete and untouched.

`chordlab.data` stores the record number and the published MD5 values so you do not have to type them.

In [3]:
print("Zenodo record :", data.ZENODO_RECORD, "  DOI:", data.ZENODO_DOI)
print("Community     :", data.ZENODO_COMMUNITY)
print("Annotations   :", data.ANNOTATIONS_REPO)
print()
files = pd.DataFrame(data.ZENODO_FILES, index=["file", "md5", "size_MB"]).T
files.index = [f"RWC-{k} ({data.SUBSET_NAMES[k]})" for k in files.index]
files

Zenodo record : 18656623   DOI: 10.5281/zenodo.18656623
Community     : https://zenodo.org/communities/rwc-music/
Annotations   : https://github.com/rwc-music/rwc-annotations.git



,file,md5,size_MB
RWC-R (Royalty-Free),RWC-R.zip,63e3b6263656a42c592ce1e90a88caa3,320
RWC-P (Popular),RWC-P.zip,960a11a2d7fb603ad0dae8428f53d4f0,4100
RWC-C (Classical),RWC-C.zip,2ac9139c4f03a65885ae0d0d299f67f8,3000
RWC-J (Jazz),RWC-J.zip,c5d7d989e1afb8257ec50a3696d90c37,2100
RWC-G (Genre),RWC-G.zip,e78cddfb6fa639bcb6a61ad873f3cceb,3900


## 1.3 Download RWC-R (about 320 MB)

The paper's first experiment uses the 15 *Royalty-Free* tracks (children's songs and folk tunes with simple, diatonic harmony). At ~320 MB it downloads in a minute or two on Colab. The function streams the zip, verifies its MD5, extracts it into `data/audio/RWC-R/` and deletes the zip.

If you already have the data, nothing is downloaded — `ensure_rwc` just returns the folder.

In [4]:
audio_dir = data.ensure_rwc("R")            # downloads + verifies on first use
wavs = sorted(audio_dir.rglob("*.wav"))
print(len(wavs), "WAV files under", audio_dir)
wavs[:3]

15 WAV files under D:\rwc-data\audio\RWC-R


[WindowsPath('D:/rwc-data/audio/RWC-R/RWC-R/RWC_R001.wav'),
 WindowsPath('D:/rwc-data/audio/RWC-R/RWC-R/RWC_R002.wav'),
 WindowsPath('D:/rwc-data/audio/RWC-R/RWC-R/RWC_R003.wav')]

In [5]:
import soundfile as sf

info = sf.info(wavs[0])
print(info)
print(f"\n{info.frames / info.samplerate:.1f} s of {info.channels}-channel {info.subtype} PCM at {info.samplerate} Hz "
      f"-> the paper's 'uncompressed 16-bit stereo PCM at 44.1 kHz'")

D:\rwc-data\audio\RWC-R\RWC-R\RWC_R001.wav
samplerate: 44100 Hz
channels: 2
duration: 02:5.149 min
format: WAV (Microsoft) [WAV]
subtype: Signed 16 bit PCM [PCM_16]

125.1 s of 2-channel PCM_16 PCM at 44100 Hz -> the paper's 'uncompressed 16-bit stereo PCM at 44.1 kHz'


## 1.4 The annotations

Annotations live in a separate GitHub repository, `rwc-music/rwc-annotations`. `ensure_annotations()` clones it (shallow, ~85 MB) into `data/rwc-annotations/`. Let us see what kinds of annotations exist for which subset — this table hides a surprise that matters for the paper.

In [6]:
ann = data.ensure_annotations()
inventory = data.annotation_inventory(ann)
inventory

,MIDI_aligned,beats,chords,melody
RWC-C,61,61,0,0
RWC-G,102,102,0,0
RWC-J,50,50,0,0
RWC-P,100,100,100,100
RWC-R,15,15,0,0


**Notice:** curated *chord* annotations exist only for **RWC-P**. RWC-R has aligned MIDI and beats, but no chord labels. The paper evaluates on RWC-R anyway — so where did its RWC-R ground truth come from? It mentions that the RWC-R MIDI files are aligned to the audio (13 of the 15, in the paper's reading of Balke et al.), which suggests the labels were derived from MIDI. The paper does not spell the procedure out, so **notebook 07** derives chord labels from the aligned MIDI explicitly and validates the procedure on RWC-P, where human labels exist. The result ships with this repository in `derived_annotations/chords_from_midi/RWC-R/`.

> This is a general lesson: when a paper's dataset section is thin, reconstructing the ground truth is part of reproducing the paper.

## 1.5 Metadata: what is actually in RWC-R?

In [7]:
meta = data.load_metadata(ann)
cols = ["Title", "Artist", "SingingLanguage", "Tempo", "LiveInstruments", "DrumInformation", "audio_start", "audio_end", "duration"]
rwc_r = meta.loc[data.track_ids("R", ann), cols]
rwc_r

,Title,Artist,SingingLanguage,Tempo,LiveInstruments,DrumInformation,audio_start,audio_end,duration
RWCID,,,,,,,,,
RWC_R001,Chou chou,Hiromi Asakawa,Japanese,120.0,"Gt, Bs, HH",Drum sequences,0.0769,123.8494,125.149093
RWC_R002,Musunde hiraite,Hiromi Asakawa,Japanese,104.0,Gt,Drum sequences,0.2220,123.9771,128.032608
RWC_R003,Akai kutsu,Hiromi Asakawa,Japanese,75.0,"Gt, Bs, HH",Drum sequences,0.2075,173.9020,176.665896
RWC_R004,Nanatsu no ko,Hiromi Asakawa,Japanese,70.0,NaN,Drum sequences,0.0244,124.5020,126.466780
RWC_R005,Momotaro,Hiromi Asakawa,Japanese,130.0,Gt,Drum sequences,0.3643,124.9477,128.353651
RWC_R006,Jingle Bells,Donna Burke,English,115.0,"Gt, Bs, HH",Drum sequences,0.0639,115.1594,118.345828
RWC_R007,Mary Had A Little Lamb,Donna Burke,English,100.0,"Gt, Bs, HH",Drum sequences,0.1286,119.2286,122.736417
RWC_R008,I've Been Workin' On The Railroad,Jeff Manning,English,130.0,"Gt, HH",Drum sequences,0.1480,119.9869,124.079433
RWC_R009,Home on the Range,Jeff Manning,English,100.0,Gt,Drum sequences,0.0189,150.9558,152.671769


In [8]:
print(f"Total duration of RWC-R: {rwc_r['duration'].sum() / 60:.1f} minutes "
      f"(the paper states 32 min 23 s)")
print("Tracks with live guitar/bass/hi-hat:", rwc_r["LiveInstruments"].notna().sum(), "of 15")

Total duration of RWC-R: 32.4 minutes (the paper states 32 min 23 s)
Tracks with live guitar/bass/hi-hat: 12 of 15


`audio_start` / `audio_end` mark where the music begins and ends inside each file; anything before or after is silence and, as we will see in notebook 08, silence is where a 24-state decoder that *always* outputs a chord loses precision.

## 1.6 A first look at the annotation files

Both formats are documented in the repository READMEs and are plain semicolon-separated text: chords as `t_start;t_end;chord` in **Harte syntax** (`Ab:min`, `Gb:maj6`, `E:7/3`, `N`), beats as `t;beat` (beat position in the bar, `1` = downbeat).

In [9]:
intervals, labels = data.load_chords("RWC_P001", ann)
print(f"RWC_P001: {len(labels)} chord segments, {len(set(labels))} distinct labels")
pd.DataFrame({"t_start": intervals[:8, 0], "t_end": intervals[:8, 1], "chord": labels[:8]})

RWC_P001: 134 chord segments, 29 distinct labels


,t_start,t_end,chord
0,0.000,0.104,N
1,0.104,1.858,Ab:min
2,1.858,3.646,Gb:maj
3,3.646,5.387,E:maj
4,5.387,9.067,Gb:maj6
5,9.067,10.762,Eb:maj
6,10.762,12.527,Ab:min
7,12.527,14.199,E:maj


In [10]:
beat_times, beat_pos = data.load_beats("RWC_R007", ann)
print(f"RWC_R007 ({meta.loc['RWC_R007', 'Title']}): {len(beat_times)} beats, "
      f"median beat period {np.median(np.diff(beat_times)):.3f} s "
      f"-> {60 / np.median(np.diff(beat_times)):.0f} BPM (metadata says {meta.loc['RWC_R007', 'Tempo']:.0f})")
print("first beats:", np.round(beat_times[:8], 3), "positions:", beat_pos[:8])

RWC_R007 (Mary Had A Little Lamb): 203 beats, median beat period 0.600 s -> 100 BPM (metadata says 100)
first beats: [0.19 0.79 1.39 1.99 2.59 3.19 3.79 4.39] positions: [1 2 3 4 1 2 3 4]


## 1.7 RWC-P is 4.1 GB — three ways to handle it

Notebook 09 needs the 100 *Popular* tracks. You have options:

1. **Download everything once** (`data.download_rwc("P")`) and keep it on Google Drive (see the optional cell at the top). Colab's disk is large enough, but the download is repeated every session unless cached.
2. **Fetch single tracks.** Zenodo serves HTTP *byte ranges*, so a zip member can be extracted without downloading the whole archive: `data.fetch_single_track("RWC_P031")` (needs `pip install remotezip`).
3. **Work locally** on a machine with the data and only use Colab for the smaller experiments.

The cell below is commented out on purpose — run it only when you get to notebook 09.

In [11]:
# data.download_rwc("P")                      # ~4.1 GB, verified by MD5
# data.fetch_single_track("RWC_P031")         # one song via HTTP range requests (pip install remotezip)

## 1.8 Data etiquette checklist

- [ ] I know the license (CC BY-NC 4.0) and my use is non-commercial.
- [ ] I cite Goto et al. (2002, 2003), Goto (2006), Müller/Balke/Goto (2025) and Balke et al. (2026) — copy them from `references.bib`.
- [ ] I never commit audio to a Git repository (size **and** license); I download at runtime instead.
- [ ] I record which *version* of the data I used (Zenodo record `18656623` = v2; annotations commit hash from `git -C data/rwc-annotations rev-parse HEAD`).

## Exercises

1. Zenodo hosts other chord datasets. Search the site for *"Schubert Winterreise Dataset"* and note its license and what annotations it contains. Could it replace RWC-R in the paper's experiment?
2. `mirdata` has loaders for the RWC subsets. Install it, look at `mirdata.initialize("rwc_popular")` and find out whether its index points at the AIST annotation files, the new curated ones, or neither.
3. Compute, from `metadata.csv`, how many RWC-P tracks have *live drums* versus *drum sequences*. Keep the numbers — the paper blames "percussive saturation" for failures and we will test that in notebook 09.